In [1]:
# Import necessary libraries from src and other dependencies
import json
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from stentfit import mesh_skeleton_beams
from stentfit.generate_artery import generate_artery_for_stent

# BeamMe: represent the artery centreline as a Cosserat curve and warp the stent onto it
from beamme.core.rotation import Rotation
from beamme.cosserat_curve.cosserat_curve import CosseratCurve
from beamme.cosserat_curve.warping_along_cosserat_curve import warp_mesh_along_curve

# STENT FEATURE EXTRACTION

In [2]:
STENT_NAME  = "stent01"     
#STENT_NAME = '2crownCrimpedXienceStent'
#STENT_NAME = '16crownCrimpedXienceStent+extremaSupports'

VERSION     = None               # e.g. "v03" -> folder "stent04_v03"; None -> "stent04"

# Output layout — everything lives under <repo>/outputs/:
#   stent_skeleton/<run>  : read the skeleton result produced by stent_skeleton.ipynb (per-stent, versioned)
#   simulation/input      : write the 4C input .yaml files (artery solid, warped stent, assembly)
#   simulation/output     : write the .vtu meshes for ParaView / 4C results
# The simulation folders are shared (not per-stent): each run overwrites the previous files.
REPO_ROOT     = Path("/Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT")
SKELETON_ROOT = REPO_ROOT / "outputs" / "stent_skeleton"
SIM_INPUT_DIR  = REPO_ROOT / "outputs" / "simulation" / "input"
SIM_OUTPUT_DIR = REPO_ROOT / "outputs" / "simulation" / "output"

run_folder = f"{STENT_NAME}_{VERSION}" if VERSION else STENT_NAME
stent_dir  = SKELETON_ROOT / run_folder

if not stent_dir.is_dir():
    available = sorted(p.name for p in SKELETON_ROOT.iterdir() if p.is_dir()) if SKELETON_ROOT.is_dir() else []
    raise FileNotFoundError(
        f"Stent folder not found: {stent_dir}\nAvailable folders: {available}"
    )

SIM_INPUT_DIR.mkdir(parents=True, exist_ok=True)
SIM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Load the stent features + skeleton directly from the folder ------------
with open(stent_dir / "stent_features.json") as f:
    features = json.load(f)                       # flat dict of geometric features

stent_centerline_dir = np.array(features["stent_centerline_direction"])   # stent axis (unit vector)

stent_skel = pd.read_csv(stent_dir / "skeleton_points.csv")
stent_skel["neighbor_ids"] = stent_skel["neighbor_ids"].apply(ast.literal_eval)

stent_length          = features["length"]
stent_diameter        = features["diameter"]
stent_r_outer         = features["r_outer"]
stent_strut_thickness = features["strut_thickness"]
stent_z_min, stent_z_max    = features["z_min"], features["z_max"]

print("Stent features")
print("--------------")
print(f"Loaded stent result from : {stent_dir.resolve()}")
print(f"Simulation input dir     : {SIM_INPUT_DIR}")
print(f"Simulation output dir    : {SIM_OUTPUT_DIR}")
print(f"Skeleton nodes           : {len(stent_skel):,}")
print(f"Centreline direction     : {stent_centerline_dir.round(4)}")
print(f"  length          : {stent_length:8.3f} mm")
print(f"  diameter        : {stent_diameter:8.3f} mm")
print(f"  r_outer         : {stent_r_outer:8.3f} mm")
print(f"  strut_thickness : {stent_strut_thickness:8.3f} mm")
print(f"  z range         : [{stent_z_min:.3f}, {stent_z_max:.3f}] mm")
print(f"  sampled points  : {features['num_points']:,}")
print(f"  skeleton nodes  : {len(stent_skel):,}")

Stent features
--------------
Loaded stent result from : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/stent_skeleton/stent01
Simulation input dir     : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input
Simulation output dir    : /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/output
Skeleton nodes           : 70,165
Centreline direction     : [1.e-04 1.e+00 0.e+00]
  length          :   18.094 mm
  diameter        :    3.152 mm
  r_outer         :    1.572 mm
  strut_thickness :    0.107 mm
  z range         : [-9.048, 9.045] mm
  sampled points  : 1,824,400
  skeleton nodes  : 70,165


# ARTERY GENERATION

In [3]:
# Generate a pipe-like artery by using the functions in generate_artery.py. Create it such that the user can specify the type

# --- Config: artery geometry -------------------------------------------------
ARTERY_TYPE     = "straight"   # "straight" | "curved" | "s_bend"
WALL_THICKNESS  = 0.5          # artery wall thickness [mm] for the 3D solid (0 = lumen surface only)
NOISE_AMPLITUDE = 0.1        # fractional wall roughness (0 = smooth pipe)
NOISE_SEED      = 42
BEND_ANGLE_DEG  = 180.0         # only used by "curved" / "s_bend"

# generate_artery_for_stent expects features wrapped as {key: {"value": ...}};
# our folder JSON is flat, so build a tiny wrapped view of the scalar features.
stent_feat_w = {k: {"value": v} for k, v in features.items() if isinstance(v, (int, float))}

artery_geometry, artery_cl, artery_radius = generate_artery_for_stent(
    stent_feat_w,
    artery_type=ARTERY_TYPE,
    wall_thickness=WALL_THICKNESS,
    noise_amplitude=NOISE_AMPLITUDE,
    noise_seed=NOISE_SEED,
    bend_angle_deg=BEND_ANGLE_DEG,
)

Artery type      : straight
Artery radius    : 2.572 mm (lumen)
Wall thickness   : 0.500 mm  (outer radius 3.072 mm)
Noise amplitude  : 0.1 (10% of radius)  seed=42
Arc length       : 27.14 mm  (stent 18.09 mm = 67% of artery)
Centreline       : 150 points  bounds [0. 0. 0.] → [ 0.    0.   27.14]
Mesh             : 19,204 vertices  38,400 faces  watertight=True


#  STENT MESHING AND ALIGNMENT

In [4]:
# Config: beam discretisation 
D_BEAM = stent_strut_thickness 
SOLID_ELEMENT_SIZE = D_BEAM * 1.5
L_EL = SOLID_ELEMENT_SIZE * 1.2

print('Element sizing for meshing:')
print(f'  Solid element size: {SOLID_ELEMENT_SIZE:.2f} mm')
print(f'  Beam element length: {L_EL:.2f} mm')
print(f'  Beam cross-section diameter: {D_BEAM:.2f} mm')

YOUNGS_MODULUS = 2.0e5
POISSON_RATIO = 0.3
DENSITY = 0.0
BEAM_CLASS_LABEL = 'Beam3rHerm2Line3'

# 1. Build the straight stent as a BeamMe beam mesh from the fitted splines in the folder.
#    (mesh_skeleton_beams rebuilds each skeleton_splines.json curve as a splinepy BSpline and
#     meshes it into one BeamMe Mesh; beam radius = strut_thickness / 2.)

beam_mesh = mesh_skeleton_beams(input_dir=str(stent_dir), output_dir=str(SIM_INPUT_DIR),
                                l_el=L_EL,
                                youngs_modulus=YOUNGS_MODULUS, 
                                poisson_ratio=POISSON_RATIO,
                                density=DENSITY,
                                beam_class_label=BEAM_CLASS_LABEL)

# 2. Represent the artery centreline as a Cosserat curve.
curve = CosseratCurve(artery_cl)

# 3. Warp the straight stent onto the curve.
#    The stent skeleton runs along +Z, so the reference triad's first basis vector must be +Z.
#    `origin` only sets the arc-length offset (it cancels out of the final position): we map the
#    stent's z-centre onto the artery's mid-arc so the stent sits centred within the vessel.
ref_rot   = Rotation([0.0, 1.0, 0.0], -np.pi / 2.0)          # first basis vector -> +Z (stent axis)
total_arc = np.linalg.norm(np.diff(artery_cl, axis=0), axis=1).sum()
z_center  = 0.5 * (features["z_min"] + features["z_max"])
origin    = np.array([0.0, 0.0, z_center - total_arc / 2.0])

warp_mesh_along_curve(beam_mesh, curve, origin=origin, reference_rotation=ref_rot)

# Warped node coordinates, for inspection and downstream use.
stent_warped = np.array([node.coordinates for node in beam_mesh.nodes])
print(f"Warped {len(stent_warped):,} beam nodes onto the artery centreline "
      f"({len(beam_mesh.elements):,} beam elements).")
print(f"  artery arc      : {total_arc:.2f} mm")
print(f"  warped stent z  : [{stent_warped[:, 2].min():.2f}, {stent_warped[:, 2].max():.2f}] mm")

# Save the warped stent beam mesh as a 4C .yaml (same format as the artery solid).
# A BeamMe Mesh dumps via an InputFile (there is no Mesh.to_yaml); validate=False since
# this is a mesh-only file (no solver sections yet).
from beamme.four_c.input_file import InputFile
stent_yaml = SIM_INPUT_DIR / "stent_warped.4C.yaml"
stent_input = InputFile()
stent_input.add(beam_mesh)
stent_input.dump(str(stent_yaml), validate=False, add_footer_application_script=False)
print(f"[saved] {stent_yaml}")

Element sizing for meshing:
  Solid element size: 0.16 mm
  Beam element length: 0.19 mm
  Beam cross-section diameter: 0.11 mm
[folder] read stent_features.json from /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/stent_skeleton/stent01
[folder] strut_thickness = 0.1073 mm

Meshed 135/135 curves (0 skipped) into a 1D beam mesh:
  2,167 nodes, 1,016 Beam3rHerm2Line3 elements
  cross-section radius 0.0536 mm, target element length 0.1931 mm
Warped 2,167 beam nodes onto the artery centreline (1,016 beam elements).
  artery arc      : 27.14 mm
  warped stent z  : [4.59, 22.56] mm
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/stent_warped.4C.yaml


# ARTERY MESHING

In [5]:
# Mesh the artery WALL into a 3D solid with GMSH and write it as a 4C .yaml.
from stentfit import mesh_artery_gmsh

MESH_TYPE = "HEX8"    # "TET4" | "TET10" | "HEX8"
                      #   TET4  : linear tets (fast; fine for the placeholder material)
                      #   TET10 : quadratic tets (use for the near-incompressible HGO material)
                      #   HEX8  : structured hexes (matches the papers; fewest elements)
  # target element edge length [mm]. Smaller -> finer mesh
                            # (tune vs. beam length: solid/beam element ratio ~2.5-5)
ARTERY_YOUNGS = 2.0         # artery solid Young's modulus [MPa] (soft tissue ~1-3 MPa; placeholder
                            # StVK for now). Used both for the mesh material and the coupling check.

artery_solid_yaml = mesh_artery_gmsh(
    r_inner=artery_radius,                    # lumen radius
    r_outer=artery_radius + WALL_THICKNESS,   # outer wall radius
    centreline=artery_cl,                     # follows straight / curved / s_bend
    mesh_type=MESH_TYPE,
    element_size=SOLID_ELEMENT_SIZE,
    noise_amplitude=NOISE_AMPLITUDE,          # same wall roughness as the artery preview
    noise_seed=NOISE_SEED,
    youngs_modulus=ARTERY_YOUNGS,
    out_path=SIM_INPUT_DIR / "artery_solid.4C.yaml",
)

[gmsh] artery wall meshed (HEX8): r_inner=2.572 r_outer=3.072 arc length=27.140 mm  (element size 0.161 mm)
[gmsh] 76,162 nodes, 56,784 HEX8 elements  noise=0.1, warped onto centreline
[gmsh] surface node sets: lumen=19040 inlet=448 outlet=448
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/artery_solid.4C.yaml


In [6]:
# Inspect: the warped stent placed inside the artery.
# The stent IS a connected 1D beam mesh (Beam3rHerm2Line3): draw each element as a polyline
# (node0 -> node1(mid) -> node2) rather than loose points, so the wireframe is visible.
elem_coords = np.array([[n.coordinates for n in el.nodes] for el in beam_mesh.elements])  # (n_el, 3, 3)
seg = np.full((len(elem_coords), 4, 3), np.nan)   # 3 nodes + a NaN gap to break the line between elements
seg[:, :3, :] = elem_coords
beam_lines = seg.reshape(-1, 3)

verts = np.asarray(artery_geometry.vertices)
faces = np.asarray(artery_geometry.faces)

fig = go.Figure()
fig.add_trace(go.Mesh3d(
    x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
    i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
    color="lightpink", opacity=0.25, name="artery", showscale=False,
))
fig.add_trace(go.Scatter3d(
    x=artery_cl[:, 0], y=artery_cl[:, 1], z=artery_cl[:, 2],
    mode="lines", line=dict(color="gray", width=3, dash="dash"), name="centreline",
))
fig.add_trace(go.Scatter3d(
    x=beam_lines[:, 0], y=beam_lines[:, 1], z=beam_lines[:, 2],
    mode="lines", line=dict(color="crimson", width=2), name="stent beams",
))
fig.update_layout(
    title=f"Stent warped onto {ARTERY_TYPE} artery — {STENT_NAME} "
          f"({len(beam_mesh.elements):,} beam elements)",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

In [7]:
# Assemble the artery solid + the warped stent beams into one BeamMe mesh and write the 4C input.
# import_artery_solid() reads the GMSH 4C .yaml solid; assemble_beam_solid() adds the beams and ties
# them to the lumen surface with a beam-to-solid coupling condition, then dumps the combined .4C.yaml.
from stentfit import import_artery_solid, assemble_beam_solid

if artery_solid_yaml is None:
    print("No artery solid mesh from the previous cell — skipping assembly.")
else:
    input_file, solid = import_artery_solid(artery_solid_yaml)

    out_path = SIM_INPUT_DIR / "artery_stent.4C.yaml"
    input_file, full_mesh = assemble_beam_solid(
        input_file, solid, beam_mesh,
        lumen_surface_index=0,        # DSURFACE 1 = lumen (written first by the mesher)
        # bc_type defaults to beam_to_solid_surface_meshtying; switch to
        # bme.bc.beam_to_solid_surface_contact for a real deployment simulation.
        output_path=out_path,
    )
    print(f"\nWrote assembled beam-to-solid 4C input file -> {out_path}")
    print("Next: materials (HGO-C artery), boundary conditions, expansion driver, solver, run 4C.")

[import] solid: 76,162 nodes, 56,784 elements, 3 surface set(s)
[assemble] coupling id 0: 1,016 beam elements <-> lumen surface (BoundaryCondition.beam_to_solid_surface_meshtying)
[assemble] combined mesh: 78,329 nodes, 57,800 elements
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/artery_stent.4C.yaml

Wrote assembled beam-to-solid 4C input file -> /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/artery_stent.4C.yaml
Next: materials (HGO-C artery), boundary conditions, expansion driver, solver, run 4C.


In [8]:
# Export the two meshes as SEPARATE files for ParaView (beam + solid).
if artery_solid_yaml is None:
    print("No assembled mesh — run the meshing / assembly cells first.")
else:
    # BeamMe writes the beam and solid parts to SEPARATE .vtu files.
    full_mesh.write_vtk(output_name="artery_stent_mesh", output_directory=str(SIM_INPUT_DIR))
    print(f"[vtk] {SIM_OUTPUT_DIR / 'artery_stent_mesh_beam.vtu'}")
    print(f"[vtk] {SIM_OUTPUT_DIR / 'artery_stent_mesh_solid.vtu'}")
    print("Open the .vtu files in ParaView to inspect the meshes.")

[vtk] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/output/artery_stent_mesh_beam.vtu
[vtk] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/output/artery_stent_mesh_solid.vtu
Open the .vtu files in ParaView to inspect the meshes.


# COUPLING ASSUMPTION CHECK

In [9]:
# Verify the mixed-dimensional beam-to-solid coupling assumptions (Steinbrecher et al.) BEFORE
# building the 4C simulation input. A later simulation-file step should gate on `coupling_ok`.
from stentfit import check_coupling_assumptions

# Actual mean beam element length (chord, end-to-end) from the meshed beams.
beam_elem_len = float(np.mean([
    np.linalg.norm(np.asarray(el.nodes[-1].coordinates) - np.asarray(el.nodes[0].coordinates))
    for el in beam_mesh.elements
]))

coupling = check_coupling_assumptions(
    beam_youngs=YOUNGS_MODULUS,             # stent E [MPa]
    solid_youngs=ARTERY_YOUNGS,             # artery E [MPa]
    beam_diameter=stent_strut_thickness,    # beam cross-section diameter (= 2 x beam radius)
    beam_element_length=beam_elem_len,      # actual mean beam element length [mm]
    solid_element_length=SOLID_ELEMENT_SIZE,
)
coupling_ok = coupling["all_passed"]
if not coupling_ok:
    print("\n[!] Coupling assumptions not satisfied — retune L_EL / SOLID_ELEMENT_SIZE (and the moduli) "
          "before building the simulation input.")

Mixed-dimensional coupling assumption check
-------------------------------------------
  [PASS] stiffness              E_beam/E_solid = 100000.0 (>= 10.0) - beam is much stiffer than the solid
  [PASS] rule_of_thumb          L_solid/D_beam = 1.50 (>= 1) - solid element >= beam cross-section diameter
  [PASS] element_length_ratio   L_beam/L_solid = 1.23 - in the optimal 1-6 band
  => ALL CHECKS PASSED


# PHYSICAL FIT CHECK

In [10]:
# Geometric compatibility of the placed stent inside the artery lumen: length fill, delivery/crimp,
# containment (all nodes inside), wall clearance, and bending strain at the artery curvature.
# NOTE: containment/clearance need a single closed LUMEN surface, so regenerate the artery with
# wall_thickness=0 (same shape/roughness) — the two-shell walled mesh confuses the inside test.
from stentfit import check_stent_artery_fit


fit = check_stent_artery_fit(
    skel_mapped=stent_warped,     # warped stent beam node positions
    skel=stent_skel,              # skeleton table (node_type / neighbours)
    features=stent_feat_w,        # wrapped features {key: {"value": ...}}
    artery_mesh=artery_geometry,       # single-wall lumen surface
    artery_cl=artery_cl,
    artery_radius=artery_radius,  # lumen radius
)

print("\nPhysical fit check")
print("------------------")
for name in ("length", "delivery", "containment", "clearance", "bending_strain"):
    c = fit[name]
    print(f"  [{'PASS' if c['passed'] else 'FAIL'}] {name:14s} {c['note']}")
fit_ok = all(fit[n]["passed"] for n in ("length", "delivery", "containment", "clearance", "bending_strain"))
print(f"  => {'ALL FIT CHECKS PASSED' if fit_ok else 'ONE OR MORE FIT CHECKS FAILED'}")

  [3/5] Containment + [4/5] Clearance … 100.0% inside, min clearance 1.048 mm, 0 penetrating

Physical fit check
------------------
  [PASS] length         Stent 18.09 mm fills 66.7% of 27.14 mm artery arc
  [PASS] delivery       Crimped OD 3.145 mm < artery ID 5.145 mm (38.9% radial margin)
  [PASS] containment    100.0% of 2,167 sampled nodes inside artery lumen (radial distance < 2.572 mm)
  [PASS] clearance      Min lumen clearance 1.048 mm (strut radius 0.054 mm), 0 penetrating nodes
  [PASS] bending_strain Artery is straight — no curvature constraint
  => ALL FIT CHECKS PASSED


# SIMULATION INPUT

In [11]:
# Build the runnable 4C simulation input: static solver + boundary conditions + a radial
# "balloon" expansion force on the stent, on top of the assembled beam-to-solid mesh.
# Gated on the coupling + physical-fit checks. Smoke test: placeholder material + meshtying
# (tied). The file is schema-validated here; running it needs a 4C binary on Linux.
from stentfit import build_smoketest_input

N_STEPS         = 10        # quasi-static load steps (the force ramps 0 -> full over the time)
EXPANSION_FORCE = 1e-4      # radial outward force per stent node at full load [N] (smoke-test value)

if not (coupling_ok and fit_ok):
    print("[skip] Coupling and/or physical-fit checks failed — fix those before building the simulation.")
else:
    simulation_yaml = build_smoketest_input(
        full_mesh, beam_mesh, artery_cl,
        out_path=SIM_INPUT_DIR / "simulation.4C.yaml",
        n_steps=N_STEPS,
        expansion_force=EXPANSION_FORCE,
    )
    print(f"\nSimulation input ready -> {simulation_yaml}")
    print("Phase B: switch meshtying -> contact, HGO-C artery material, elasto-plastic stent, ~100 steps.")

[sim] static smoke test: 10 steps, radial expansion force 0.0001 N ramped over 1,151 stent nodes
[sim] BCs: artery inlet+outlet fixed, one stent node pinned; coupling = beam-to-solid meshtying (tied)
[saved] /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/simulation.4C.yaml
[sim] schema-validated. Run in 4C on Linux: set BEAMME_FOUR_C_EXE and launch 4C on this file.

Simulation input ready -> /Users/vural/Desktop/RWTH - SiSc/Projects/Github/stentFIT/outputs/simulation/input/simulation.4C.yaml
Phase B: switch meshtying -> contact, HGO-C artery material, elasto-plastic stent, ~100 steps.
